# Truth-labelled Random Forest: all-$p_T$ smoke study and event methodology

> ## Scope warning — not thesis evidence
> `high20` is an **all-$p_T$ smoke test** built from the currently available high-threshold sources. It is **not low-$p_T$ thesis evidence**: its manifest declares `pt_region: all`, and an authoritative thesis run requires a coherent truth-labelled sample with $p_T < 20$ GeV.

This presentation notebook re-scores the frozen RF model on the existing event-grouped train/validation/test Parquet splits. Four operating points (WPs) are selected on validation once, then applied unchanged to test.

## Reproducibility warning — version mismatch

> The historical `rf_model.joblib` was serialized with **scikit-learn 1.9.0**, while this notebook runs with **scikit-learn 1.8.0**. scikit-learn emits its `InconsistentVersionWarning` when the model is loaded; this notebook deliberately does **not** suppress that warning.
>
> Consequently, every number here is a **smoke-study evaluation only**, not a version-reproducible result or thesis-grade evidence. Reproduce the model under its original software environment before making any scientific claim from these scores.

## Method in one slide

1. Per reconstructed jet, the RF estimates a primary **b-vs-all** score using truth-labelled b/c/g/uds jets.
2. Validation selects four frozen WPs: 80%/90% b purity and 80%/90% b efficiency. Test is never used to choose a threshold.
3. Object metrics report b efficiency/purity and flavour-specific c, g, and uds mistags.
4. Event metrics group scored candidates by `global_event_id`: any-b event efficiency/purity, per-event jet metrics, zero-selection rate, and top-$k$ ranking.

The RF remains the primary b-vs-all tagger. The companion hybrid smoke design is a bounded conditional b-vs-c reranking study around an RF gate; it must never interpret g/uds as c or replace the primary score.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'ml').is_dir():
    ROOT = next(parent for parent in (ROOT, *ROOT.parents) if (parent / 'ml').is_dir())
DATASET_DIR = ROOT / 'outputs' / 'high20_jet_flavor'
MODEL_PATH = ROOT / 'outputs' / 'rf_high20_jet_flavor' / 'rf_model.joblib'
OUTPUT_DIR = ROOT / 'outputs' / 'rf_high20_event_smoke_evaluated'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.evaluation import WORKING_POINTS, evaluate_working_points
from ml.train_rf import select_features

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = {'b': '#1f77b4', 'c': '#ff7f0e', 'g': '#2ca02c', 'uds': '#9467bd'}
print(f'Repository: {ROOT}')
print(f'Generated artifacts: {OUTPUT_DIR}')

Repository: /home/juanm/Btagginghep
Generated artifacts: /home/juanm/Btagginghep/outputs/rf_high20_event_smoke_evaluated


In [2]:
import platform
import sklearn

SERIALIZED_SKLEARN_VERSION = '1.9.0'
RUNTIME_VERSIONS = {
    'python': platform.python_version(),
    'scikit_learn': sklearn.__version__,
    'model_serialized_scikit_learn': SERIALIZED_SKLEARN_VERSION,
}
display(pd.Series(RUNTIME_VERSIONS, name='Reproducibility versions'))
print('WARNING: Model serialization and runtime scikit-learn versions differ; do not suppress the loader warning.')

python                           3.14.7
scikit_learn                      1.8.0
model_serialized_scikit_learn     1.9.0
Name: Reproducibility versions, dtype: object

In [3]:
manifest_path = DATASET_DIR / 'dataset_manifest.json'
if not manifest_path.is_file() or not MODEL_PATH.is_file():
    raise FileNotFoundError('Expected manifest or frozen RF model is missing; run from the repository checkout.')

manifest = json.loads(manifest_path.read_text())
splits = {name: pd.read_parquet(DATASET_DIR / f'{name}.parquet') for name in ('train', 'val', 'test')}
model = joblib.load(MODEL_PATH)
features = select_features(splits['train'])
missing_features = sorted(set(features) - set(getattr(model, 'feature_names_in_', features)))
if missing_features:
    raise ValueError(f'Frozen model does not expose selected features: {missing_features}')

summary = pd.DataFrame({
    'split': list(splits),
    'jets': [len(frame) for frame in splits.values()],
    'events': [frame['global_event_id'].nunique() for frame in splits.values()],
    'b jets': [int((frame['sample_label'] == 'b').sum()) for frame in splits.values()],
})
display(summary)
print('Manifest pT region:', manifest['pt_region'], '| expression:', manifest['pt_expression'])
print('Feature count from current ml.train_rf.select_features:', len(features))
print('Generator max_jets_per_event:', manifest['generator_settings'].get('max_jets_per_event'))

/home/juanm/.local/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/juanm/.local/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,split,jets,events,b jets
0,train,255114,170393,63942
1,val,54936,36513,13761
2,test,54699,36513,13658


Manifest pT region: all | expression: all jet pT values
Feature count from current ml.train_rf.select_features: 32
Generator max_jets_per_event: 4


## Companion semantics and the current limitation

**Per-jet evaluation** uses only scored rows with explicit b/c/g/uds truth. This is the correct denominator for object b efficiency, purity, and flavour mistags.

**All-reconstructed-event evaluation** should instead score every reconstructed companion jet from the same original event, including candidates without recognized truth. Candidate selection and top-$k$ then use all scored jets; truth-dependent denominators remain restricted to known truth. This prevents unknown companions from being silently counted as non-b.

The historical `high20_jet_flavor` dataset has no `val_companion.parquet` or `test_companion.parquet`. Therefore this notebook honestly evaluates events over the supervised test rows only; it does **not** claim all-reconstructed-event coverage. The manifest also records the historical four-jet extraction cap.

In [4]:
companion_paths = {name: DATASET_DIR / f'{name}_companion.parquet' for name in ('val', 'test')}
available_companions = {name: path.is_file() for name, path in companion_paths.items()}
event_score_source = 'test_companion' if available_companions['test'] else 'supervised test rows (historical limitation)'
display(pd.DataFrame({'split': list(available_companions), 'companion_available': list(available_companions.values())}))
print('Event candidate source:', event_score_source)

,split,companion_available
0,val,False
1,test,False


Event candidate source: supervised test rows (historical limitation)


In [5]:
def keyed_scores(frame: pd.DataFrame, scores: np.ndarray) -> pd.DataFrame:
    columns = [name for name in ('global_event_id', 'root_file', 'event_in_file', 'jet_rank', 'source_sample_label', 'sample_label', 'jet_flavor', 'is_b') if name in frame]
    result = frame.loc[:, columns].copy()
    result['score_rf'] = scores
    return result

def score(frame: pd.DataFrame) -> np.ndarray:
    return model.predict_proba(frame.loc[:, features])[:, 1]

val_scores = keyed_scores(splits['val'], score(splits['val']))
test_scores = keyed_scores(splits['test'], score(splits['test']))
val_scores.to_csv(OUTPUT_DIR / 'validation_scores.csv', index=False)
test_scores.to_csv(OUTPUT_DIR / 'test_scores.csv', index=False)

event_scores = test_scores
if available_companions['test']:
    test_companion = pd.read_parquet(companion_paths['test'])
    event_scores = keyed_scores(test_companion, score(test_companion))
    event_scores.to_csv(OUTPUT_DIR / 'test_companion_scores.csv', index=False)

def select_frozen_working_points(validation: pd.DataFrame) -> dict:
    """Tie-preserving O(n log n) selection equivalent to the frozen-WP definition."""
    truth = validation['jet_flavor'].abs().map({5: 'b', 4: 'c', 1: 'uds', 2: 'uds', 3: 'uds', 21: 'g'})
    known = validation.loc[truth.notna(), ['score_rf']].copy()
    known['truth'] = truth.loc[known.index]
    grouped = known.assign(is_b=known['truth'].eq('b')).groupby('score_rf', sort=False).agg(total=('truth', 'size'), b=('is_b', 'sum')).sort_index(ascending=False)
    grouped['selected'] = grouped['total'].cumsum()
    grouped['selected_b'] = grouped['b'].cumsum()
    grouped['efficiency'] = grouped['selected_b'] / grouped['b'].sum()
    grouped['purity'] = grouped['selected_b'] / grouped['selected']
    report = {}
    for name, (criterion, target) in WORKING_POINTS.items():
        eligible = grouped.loc[grouped[criterion] >= target]
        if eligible.empty:
            report[name] = {'criterion': criterion, 'target': target, 'feasible': False, 'threshold': None}
            continue
        chosen = eligible.index.max() if criterion == 'efficiency' else eligible.index.min()
        row = grouped.loc[chosen]
        report[name] = {'criterion': criterion, 'target': target, 'feasible': True, 'threshold': float(chosen), 'validation_b_efficiency': float(row['efficiency']), 'validation_b_purity': float(row['purity'])}
    return report

working_points = select_frozen_working_points(val_scores)
report = evaluate_working_points(test_scores, working_points, 'score_rf', event_scores=event_scores)
(OUTPUT_DIR / 'working_points.json').write_text(json.dumps({
    'threshold_source': 'validation',
    'event_score_source': event_score_source,
    'working_points': report,
}, indent=2) + '\n')

wp_rows = []
for name, values in report.items():
    if values['object'] is not None:
        wp_rows.append({'working_point': name, **values['definition'], **values['object']})
object_table = pd.DataFrame(wp_rows)
event_table = pd.DataFrame([{'working_point': name, **values['event']} for name, values in report.items() if values['event'] is not None])
object_table.to_csv(OUTPUT_DIR / 'object_working_points.csv', index=False)
event_table.to_csv(OUTPUT_DIR / 'event_working_points.csv', index=False)
display(object_table[['working_point', 'threshold', 'b_efficiency', 'b_purity', 'c_mistag', 'g_mistag', 'uds_mistag']])

/home/juanm/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


/home/juanm/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


,working_point,threshold,b_efficiency,b_purity,c_mistag,g_mistag,uds_mistag
0,purity_80,0.800369,0.520940,0.799888,0.140501,0.019824,0.014801
1,purity_90,0.892170,0.330283,0.893975,0.043429,0.006033,0.003919
2,efficiency_80,0.599272,0.805316,0.618825,0.421165,0.109558,0.086592
3,efficiency_90,0.519269,0.907087,0.538605,0.574122,0.188565,0.162671


In [6]:
names = object_table['working_point'].tolist()
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
ax.bar(x - 0.25, object_table['b_efficiency'], 0.25, label='Object b efficiency', color='#1f77b4')
ax.bar(x, object_table['b_purity'], 0.25, label='Object b purity', color='#2ca02c')
ax.bar(x + 0.25, event_table.set_index('working_point').loc[names, 'any_b_event_efficiency'], 0.25, label='Any-b event efficiency', color='#d62728')
ax.set_xticks(x, names, rotation=12)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Metric')
ax.set_title('Frozen validation WPs applied to the test split')
ax.legend(ncol=3, loc='upper center')
fig.savefig(OUTPUT_DIR / 'object_and_event_wp_metrics.png', dpi=220, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
for offset, label in zip((-0.25, 0, 0.25), ('c', 'g', 'uds')):
    ax.bar(x + offset, object_table[f'{label}_mistag'], 0.25, label=f'{label} mistag', color=PALETTE[label])
ax.set_xticks(x, names, rotation=12)
ax.set_yscale('log')
ax.set_ylabel('Mistag probability (log scale)')
ax.set_title('Flavour-specific mistags at frozen test WPs')
ax.legend(ncol=3)
fig.savefig(OUTPUT_DIR / 'flavour_mistags_at_wps.png', dpi=220, bbox_inches='tight')
plt.show()

/home/juanm/singularity-tmp/ipykernel_2317002/811385956.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/home/juanm/singularity-tmp/ipykernel_2317002/811385956.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
event_plot_columns = ['any_b_event_efficiency', 'any_b_event_purity', 'top1_b_hit', 'top2_b_hit']
event_labels = ['Any-b efficiency', 'Any-b purity', 'Top-1 b hit', 'Top-2 b hit']
fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
width = 0.18
for index, (column, label) in enumerate(zip(event_plot_columns, event_labels)):
    values = event_table.set_index('working_point').loc[names, column].fillna(0)
    ax.bar(x + (index - 1.5) * width, values, width, label=label)
ax.set_xticks(x, names, rotation=12)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Event-level metric')
ax.set_title(f'Event metrics — candidates: {event_score_source}')
ax.legend(ncol=2)
fig.savefig(OUTPUT_DIR / 'event_level_metrics.png', dpi=220, bbox_inches='tight')
plt.show()
display(event_table[['working_point', 'candidate_events', 'known_truth_events', 'zero_selection_rate', *event_plot_columns]])

/home/juanm/singularity-tmp/ipykernel_2317002/1453094588.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,working_point,candidate_events,known_truth_events,zero_selection_rate,any_b_event_efficiency,any_b_event_purity,top1_b_hit,top2_b_hit
0,purity_80,36513,36513,0.785337,0.611117,0.786935,0.989894,0.998221
1,purity_90,36513,36513,0.872675,0.409789,0.889654,0.989894,0.998221
2,efficiency_80,36513,36513,0.600142,0.859507,0.594178,0.989894,0.998221
3,efficiency_90,36513,36513,0.497165,0.935698,0.514379,0.989894,0.998221


In [8]:
fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
for label in ('b', 'c', 'g', 'uds'):
    subset = test_scores.loc[test_scores['sample_label'] == label, 'score_rf']
    ax.hist(subset, bins=45, range=(0, 1), density=True, histtype='step', linewidth=2.0, label=label, color=PALETTE[label])
for name, threshold in object_table[['working_point', 'threshold']].itertuples(index=False):
    ax.axvline(threshold, color='#444444', alpha=0.35, linestyle='--')
ax.set_xlabel('Frozen RF b-vs-all score')
ax.set_ylabel('Density')
ax.set_title('Test score distributions by truth label')
ax.legend(title='Truth label', ncol=4)
fig.savefig(OUTPUT_DIR / 'test_score_distributions.png', dpi=220, bbox_inches='tight')
plt.show()

if hasattr(model, 'feature_importances_'):
    importance = pd.DataFrame({'feature': features, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
    top = importance.head(20).sort_values('importance')
    fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
    ax.barh(top['feature'], top['importance'], color='#1f77b4')
    ax.set_xlabel('Random Forest feature importance')
    ax.set_title('Top 20 frozen-model feature importances')
    fig.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=220, bbox_inches='tight')
    plt.show()
else:
    print('Feature importance is unavailable for this fitted model type.')

/home/juanm/singularity-tmp/ipykernel_2317002/2474556819.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/home/juanm/singularity-tmp/ipykernel_2317002/2474556819.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## What an authoritative low-$p_T$ campaign must provide

- Coherent b/c/g/uds generator samples with per-jet `Jet.Flavor` truth and an explicit $p_T < 20$ GeV selection.
- Event-disjoint train/validation/test partitions, raw per-track inputs, provenance (generator, detector, weights, hashes), and retained reconstructed candidates.
- Companion splits containing **all reconstructed jets per event**, including unknown-truth candidates, so event selection and ranking are physically complete.
- Sufficient truth statistics per class (target: 50k/10k/10k b, c, g, uds jets across train/validation/test) plus mixed b+c events (20k/4k/4k) for the conditional study.

Until that campaign exists, the outputs below are reproducibility and methodology smoke-test artifacts only.

In [9]:
hybrid_report_path = ROOT / 'outputs' / 'hybrid_bc_high20_smoke' / 'hybrid_bc_report.json'
hybrid_summary = {
    'primary_tagger': 'RF b-vs-all',
    'conditional_study': 'bounded b-vs-c reranking inside an RF gate',
    'semantic_guard': 'P(b | truth-selected b/c study), never a b-vs-all score; g/uds are excluded.',
    'mode': 'SMOKE_TEST',
    'existing_smoke_report_available': hybrid_report_path.is_file(),
}
if hybrid_report_path.is_file():
    existing_hybrid = json.loads(hybrid_report_path.read_text())
    hybrid_summary['reported_mode'] = existing_hybrid.get('mode')
    hybrid_summary['classical_controls'] = existing_hybrid.get('classical_controls')
(OUTPUT_DIR / 'methodology_summary.json').write_text(json.dumps({
    'scope': 'all-pT smoke test; not low-pT thesis evidence',
    'dataset_manifest': str(manifest_path),
    'model': str(MODEL_PATH),
    'reproducibility_warning': 'Historical model serialized with scikit-learn 1.9.0 is evaluated under a different runtime; this is smoke-study evaluation only.',
    'versions': RUNTIME_VERSIONS,
    'event_score_source': event_score_source,
    'hybrid_bc_smoke_design': hybrid_summary,
}, indent=2) + '\n')
display(pd.Series(hybrid_summary, name='Hybrid b/c smoke design'))
print('Artifacts written:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name)

primary_tagger                                                           RF b-vs-all
conditional_study                         bounded b-vs-c reranking inside an RF gate
semantic_guard                     P(b | truth-selected b/c study), never a b-vs-...
mode                                                                      SMOKE_TEST
existing_smoke_report_available                                                 True
reported_mode                                                             SMOKE_TEST
classical_controls                 [logistic_regression, rbf_svm, restricted_rf, ...
Name: Hybrid b/c smoke design, dtype: object

Artifacts written:
 - event_level_metrics.png
 - event_working_points.csv
 - feature_importance.csv
 - feature_importance.png
 - flavour_mistags_at_wps.png
 - methodology_summary.json
 - object_and_event_wp_metrics.png
 - object_working_points.csv
 - test_score_distributions.png
 - test_scores.csv
 - validation_scores.csv
 - working_points.json
